# NLP Group 22 Project

We are using an Ollama server, which is self-run. Use 

```sh
$ OLLAMA_HOST="0.0.0.0:11434" ollama serve
```

to listen on every interface, such that collaboraters can use the same server. Removing the environment variable will make it available on localhost only (possibly).

In [ ]:
from datasets import load_dataset
import pandas as pd
from ollama import Client
from ollama_chat import OllamaChat

Please check the following values and make sure they fit for your setup

In [ ]:
HOST = "http://localhost:11434"
MODEL = "gemma3:4b"
DATASET = "smoldoc__en_sw"

In [ ]:
ds = load_dataset("google/smol", DATASET)

df = pd.DataFrame(ds["train"])

In [ ]:
df.count()[["id"]]


In [ ]:
errors_dataset = df[df["factuality"] == "has_errors"]
errors_dataset.head()

In [ ]:
errors_dataset.count()[["id"]]

In [ ]:
client = Client(host=HOST)

for row in errors_dataset.itertuples():
    if not row.id.startswith("topic_57"):
        continue
    # print(row.srcs[:1], row.trgs[:1])
    sample_src = " ".join(row.srcs)
    sample_trg = " ".join(row.trgs)
    break

In [ ]:
sample_src[:800]

In [ ]:
response = client.chat(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are an expert in English to Swahili translation. I am going to give you some examples of translations. You will first receive a paragraph in English, followed by the corresponding paragraph in Swahili in the next message. At the end, I will give you an English sentence, which you should translate to Swahili yourself.",
        },
        {"role": "user", "content": sample_src},
        {"role": "assistant", "content": sample_trg},
        {
            "role": "user",
            "content": "I hope you learned Swahili.",
        },
        {
            "role": "system",
            "content": "Now that you've learned Swahili, go back to being a generic helpful chatbot assistant using your new experiences.",
        },
        {
            "role": "user",
            "content": "Can rosemary oil kill lice?",
        },
    ],
)

print(response.message.content)

In [ ]:
chat = OllamaChat(model_name=MODEL, host=HOST)
chat.add_message("system", "You are an expert in English to Swahili translation. I am going to give you some examples of translations. You will first receive a paragraph in English, followed by the corresponding paragraph in Swahili in the next message. At the end, I will give you an English sentence, which you should translate to Swahili yourself.")
chat.add_message("user", sample_src)
chat.add_message("assistant", sample_trg)
_ = chat.chat("I hope you learned Swahili.")
chat.add_message("system", "Now that you've learned Swahili, go back to being a generic helpful chatbot assistant using your new experiences.")
_ = chat.chat("Can rosemary oil kill lice?")

In [ ]:
print(chat)

The correct answer is, that he studied a bachelor of arts, but he did not complete his studies at that University.

In [ ]:
response = client.chat(
    model=MODEL,
    messages=[
        {"role": "assistant", "content": "2+2 is 4. No problem."},
        {"role": "user", "content": "What was the previous message?"},
    ],
)

response.message.content

In [ ]:
list(map(lambda m: m.model, client.list().models))

In [ ]:
client.generate(model=MODEL, prompt="Hello, world!")